In [1]:
import subprocess
subprocess.run(['pip', 'install', 'ase', 'pymatgen', '--break-system-packages', '-q'])

CompletedProcess(args=['pip', 'install', 'ase', 'pymatgen', '--break-system-packages', '-q'], returncode=0)

In [2]:
from ase.build import surface
from ase.io import read
from ase.visualize.plot import plot_atoms
import matplotlib.pyplot as plt
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.io.lammps.data import LammpsData
import os

In [3]:
alloy = read('bulk_structure/MgZn2_relaxed.cif')
slab = surface(alloy, (1, 0, 0), 4, vacuum=8)

In [4]:
print("Atom index | Species | Position (x, y, z)")
for i, atom in enumerate(slab):
    print(f"  {i:3d} | {atom.symbol:2s} | "
          f"x={atom.position[0]:.4f} "
          f"y={atom.position[1]:.4f} "
          f"z={atom.position[2]:.4f}")

Atom index | Species | Position (x, y, z)
    0 | Mg | x=2.6535 y=3.7377 z=8.7594
    1 | Mg | x=5.3210 y=4.8283 z=10.2995
    2 | Mg | x=5.3210 y=8.0761 z=10.2995
    3 | Mg | x=2.6535 y=0.4899 z=8.7594
    4 | Zn | x=2.6535 y=4.2830 z=11.8395
    5 | Zn | x=2.6535 y=8.6214 z=11.8395
    6 | Zn | x=4.0057 y=2.1138 z=11.0588
    7 | Zn | x=1.3013 y=2.1138 z=11.0588
    8 | Zn | x=5.3210 y=2.1138 z=8.7807
    9 | Zn | x=3.9688 y=6.4522 z=8.0000
   10 | Zn | x=1.3382 y=6.4522 z=8.0000
   11 | Zn | x=2.6535 y=6.4522 z=10.2781
   12 | Mg | x=5.3210 y=3.7377 z=13.3796
   13 | Mg | x=2.6535 y=4.8283 z=14.9197
   14 | Mg | x=2.6535 y=8.0761 z=14.9197
   15 | Mg | x=5.3210 y=0.4899 z=13.3796
   16 | Zn | x=5.3210 y=4.2830 z=16.4598
   17 | Zn | x=5.3210 y=8.6214 z=16.4598
   18 | Zn | x=1.3382 y=2.1138 z=15.6791
   19 | Zn | x=3.9688 y=2.1138 z=15.6791
   20 | Zn | x=2.6535 y=2.1138 z=13.4010
   21 | Zn | x=1.3013 y=6.4522 z=12.6203
   22 | Zn | x=4.0057 y=6.4522 z=12.6203
   23 | Zn | x=5.321

In [5]:
from ase.visualize import view

In [6]:
view(slab)

<Popen: returncode: None args: ['C:\\Users\\Kai Savage\\anaconda3\\python.ex...>

In [5]:
from ase.build import surface, make_supercell
from ase.io import read
from ase.visualize import view
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.io.lammps.data import LammpsData
import os

alloy = read('bulk_structure/MgZn2_relaxed.cif')
slab = surface(alloy, (1, 0, 0), 4, vacuum=8)

matrix = [[1, 0, 0],
          [0, 1, 0],
          [0, 0, 1]]
slab_doubled = make_supercell(slab, matrix)

# Move atom 40 to bottom surface
slab_doubled[41].position = [5.321, 4.283, 7.220]

# Visualise to confirm
view(slab_doubled)

# Convert to pymatgen and write .data file
adaptor = AseAtomsAdaptor()
pymatgen_slab = adaptor.get_structure(slab_doubled)

os.makedirs("slabs", exist_ok=True)
lammps_data = LammpsData.from_structure(pymatgen_slab, atom_style="atomic")
lammps_data.write_file("slabs/mgzn2_100_4layers_reconstructed_ase.data")

mg_count = sum(1 for site in pymatgen_slab if site.species_string == "Mg")
zn_count = sum(1 for site in pymatgen_slab if site.species_string == "Zn")
print(f"Mg: {mg_count} Zn: {zn_count}")
print(f"Stoichiometric: {'YES' if zn_count == 2 * mg_count else 'NO'}")

Mg: 16 Zn: 32
Stoichiometric: YES


In [6]:
from pymatgen.core.surface import Slab
from pymatgen.io.lammps.data import LammpsData

lammps_data = LammpsData.from_file(
    "slabs/mgzn2_100_4layers_reconstructed_ase.data",
    atom_style="atomic"
)
structure = lammps_data.structure

# Convert to Slab object
slab = Slab(
    structure.lattice,
    structure.species,
    structure.frac_coords,
    miller_index=(1, 0, 0),
    oriented_unit_cell=structure,
    shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]]
)

print(f"Symmetric: {slab.is_symmetric()}")

Symmetric: True


In [8]:
#unrelaxed surface energy calculation
E_slab = -61.077317    # eV
E_bulk_per_fu = -16.195843 / 4   # eV per formula unit = -4.048961 eV
n = 48 / 3                 # formula units in slab = 32
A = 46.290              # Ang^2

S_eV_per_ang2 = (E_slab - n * E_bulk_per_fu) / (2 * A)

# Convert to J/m^2 (1 eV/Ang^2 = 16.0218 J/m^2)
S_J_per_m2 = S_eV_per_ang2 * 16.0218

print(f"E_bulk per formula unit = {E_bulk_per_fu:.6f} eV")
print(f"n * E_bulk = {n * E_bulk_per_fu:.5f} eV")
print(f"E_slab - n*E_bulk = {E_slab - n * E_bulk_per_fu:.5f} eV")
print(f"S = {S_eV_per_ang2:.6f} eV/Ang^2")
print(f"S = {S_J_per_m2:.4f} J/m^2")

E_bulk per formula unit = -4.048961 eV
n * E_bulk = -64.78337 eV
E_slab - n*E_bulk = 3.70605 eV
S = 0.040031 eV/Ang^2
S = 0.6414 J/m^2


In [9]:
#relaxed surface energy calculation
E_slab_relaxed =   -61.2282728988001 # eV
E_bulk_per_fu = -16.195843 / 4  # eV per formula unit
n = 48 / 3                      # 32 formula units
A = 46.289                    # Ang^2

S_relaxed_eV = (E_slab_relaxed - n * E_bulk_per_fu) / (2 * A)
S_relaxed_Jm2 = S_relaxed_eV * 16.0218

print(f"E_slab_relaxed - n*E_bulk = {E_slab_relaxed - n * E_bulk_per_fu:.5f} eV")
print(f"S relaxed = {S_relaxed_eV:.6f} eV/Ang^2")
print(f"S relaxed = {S_relaxed_Jm2:.4f} J/m^2")

# Compare with unrelaxed
S_unrelaxed_Jm2 = 0.6414
print(f"\nUnrelaxed S = {S_unrelaxed_Jm2:.4f} J/m^2")
print(f"Relaxed S   = {S_relaxed_Jm2:.4f} J/m^2")
print(f"Relaxation energy = {S_unrelaxed_Jm2 - S_relaxed_Jm2:.4f} J/m^2")

E_slab_relaxed - n*E_bulk = 3.55510 eV
S relaxed = 0.038401 eV/Ang^2
S relaxed = 0.6153 J/m^2

Unrelaxed S = 0.6414 J/m^2
Relaxed S   = 0.6153 J/m^2
Relaxation energy = 0.0261 J/m^2
